# Philadelphia, PA — Wage Tax → Land Value Tax Swap

**What this notebook models:** eliminating Philadelphia's Wage & Earnings Tax entirely and replacing the lost revenue with a new, separate, pure land value tax (no improvement/building component). The existing real estate (property) tax system is left completely unchanged — see `model.ipynb` for that model. This is a genuinely different tax instrument being swapped in, not a rate or base change to the property tax.

**Why this deviates from the standard 7-section template** (`.claude/skills/build-notebook.md`): the wage tax has no parcel-level data — it's owed by people on their wages, not by parcels — and there is no public, FOIA-able parcel-level wage dataset. This notebook works at **census tract** granularity for the wage-tax side (the finest geography with a reliable public aggregate-income proxy) and rolls the parcel-level modeled land tax up to the same tracts for comparison. The "winners and losers" table in this notebook is therefore tract-level, not parcel-level.

**Policy assumptions:**
- **Swap structure**: Wage & Earnings Tax → $0. A new land-only tax is added on top of the unchanged existing property tax, sized via revenue-neutral solve to recoup the eliminated wage tax revenue.
- **Wage tax rates being eliminated**: 3.75% resident / 3.44% non-resident (FY2024–25).
- **Revenue target**: ~$2.51B (FY2024 estimated total Wage & Earnings Tax revenue), derived here as (ACS-modeled resident total) + (published non-resident share, extrapolated) — see Section 4.
- **Data sources**: ACS 5-year table B19062 (aggregate wage/salary income) at census tract level for the resident wage tax base; Census LODES Workplace Area Characteristics (WAC) as a job-count cross-check only (its earnings tiers are too coarse to estimate dollars); the existing `cities/philadelphia/data/parcels.gpq` OPA parcel cache (built by `model.ipynb`, read-only here) for the land tax base.

**Limitations to hold in mind while reading the results:**
1. **Mixed incidence.** A tract's current wage tax is borne by *all residents* — owners and renters alike. Its new land tax is borne only by *landowners* in that tract. "Net change" per tract blends two different incidence populations; it is not the same statement as "this household is better/worse off."
2. **The commuter transfer is a first-order finding, not a footnote.** Roughly a third of current wage tax revenue is paid by non-resident commuters, who mostly live outside city limits and so cannot be attributed to any Philadelphia tract. Eliminating the wage tax and replacing it entirely with a land tax shifts that share of the burden onto *city landowners* — a real, sizeable transfer from (mostly suburban) commuters to city landowners. See the `incidence_shift.png` chart in Section 7.
3. **Data-quality caveats.** ACS aggregate-income has genuine tract-level margin of error (carried through as `agg_wage_income_moe`). LODES earnings tiers are 3 coarse buckets and the data lags ~2 years behind the current year.
4. **Open question, not resolved here:** whether Philadelphia's Wage Tax revenue splits between the City General Fund and School District (the way the real estate tax splits 0.6317%/0.7681%). If it does, decide which figure(s) this notebook should validate against before treating the revenue-neutrality check as final — see the `PUBLISHED_TOTAL_WAGE_TAX_REVENUE` constant below.
5. **Policy default, not a given:** the new land tax reuses `full_exmp` (today's property-tax full-exemption flag) as its own exemption criterion. That's a reasonable default, but an explicit policy choice — change it in Section 5 if a different exemption treatment is wanted.

## Section 1 — Imports and Constants

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.census_utils import get_census_tracts_shapefile, match_to_census_tracts
from lvt.wage_tax_utils import (
    get_resident_wage_tax_by_tract,
    fetch_lodes_wac,
    summarize_lodes_workplace_jobs,
    compute_wage_tax_revenue_target,
    model_land_only_tax,
    save_wage_tax_swap_tract_export,
    save_wage_tax_swap_parcel_export,
)
from lvt.viz import create_wage_tax_swap_report

CITY_NAME = 'philadelphia_wage_tax_swap'  # distinct slug from model.ipynb's 'philadelphia' —
                                            # this is a different tax instrument, not a re-run
STATE_FIPS = '42'
COUNTY_FIPS = '101'
FIPS = STATE_FIPS + COUNTY_FIPS
MODEL_TYPE = 'wage_tax_swap:land_only'

# Wage & Earnings Tax parameters (City of Philadelphia, FY2024-25)
RESIDENT_WAGE_TAX_RATE = 0.0375        # residents owe this on ALL wages, regardless of work location
NONRESIDENT_WAGE_TAX_RATE = 0.0344     # non-residents owe this on wages earned working in Philadelphia
RESIDENT_REVENUE_SHARE = 0.662         # resident share of total wage tax revenue, FY2024 (Pew, Oct 2024 brief)
PUBLISHED_TOTAL_WAGE_TAX_REVENUE = 2_510_000_000  # FY2024 estimated total Wage & Earnings Tax revenue
                                                    # (Pew, Oct 2024). OPEN QUESTION: confirm whether this
                                                    # splits city General Fund vs. School District before
                                                    # treating the revenue-neutrality check as final.
ACS_YEAR = 2022
LODES_YEAR = 2021  # LODES lags roughly 2 years behind the current year

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Load Parcel Data (shared, read-only cache)

This notebook reads the parcel cache shared by all four Philadelphia notebooks (`cities/philadelphia/data/parcels.gpq`) but does not build or rebuild it — that's `model.ipynb`'s responsibility (see CLAUDE.md's cache-ownership note). Only land value and exemption status are needed here; no improvement-value imputation is required (a land-only tax has no improvement side).

In [ ]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'

if not PARCEL_PATH.exists():
    raise FileNotFoundError(
        f'{PARCEL_PATH} not found. This notebook reads the parcel cache shared by all '
        'Philadelphia notebooks but does not build it — run cities/philadelphia/model.ipynb '
        'first to fetch and cache OPA parcel data.'
    )

gdf = gpd.read_parquet(PARCEL_PATH)

REQUIRED_COLS = ['parcel_number', 'taxable_land', 'taxable_building', 'geometry']
missing = [c for c in REQUIRED_COLS if c not in gdf.columns]
if missing:
    raise ValueError(f'parcels.gpq cache is missing required columns {missing}; rebuild via model.ipynb')

print(f'Loaded {len(gdf):,} parcels from cache')

## Section 3 — Classify (exemption flag only)

Unlike `model.ipynb`, this notebook does not need the full OPA property-category mapping — the wage-tax-swap report is tract-level, not category-level. Only the full-exemption flag is needed, built identically to `model.ipynb`'s definition for consistency.

In [ ]:
gdf['taxable_land'] = pd.to_numeric(gdf['taxable_land'], errors='coerce').fillna(0.0)
gdf['taxable_building'] = pd.to_numeric(gdf['taxable_building'], errors='coerce').fillna(0.0)
gdf['taxable_total'] = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0)
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

print(f'Total parcels: {len(gdf):,}  |  Fully exempt: {gdf["full_exmp"].sum():,}')

## Section 4 — Current Wage Tax Base (census tract)

Resident wage tax liability is modeled directly from ACS aggregate wage/salary income by tract (table B19062 — not B19061, which also includes self-employment income already captured by Philadelphia's separate Net Profits Tax). Non-resident commuter liability isn't geographically attributable to a Philadelphia tract (most commuters live outside city limits) and LODES earnings tiers are too coarse to estimate dollars directly, so the non-resident total is extrapolated from the published resident/non-resident revenue split instead. LODES is used only as a job-count plausibility cross-check.

In [ ]:
wage_tax_df = get_resident_wage_tax_by_tract(
    FIPS, resident_rate=RESIDENT_WAGE_TAX_RATE, year=ACS_YEAR,
)
tract_boundaries = get_census_tracts_shapefile(FIPS)

tract_gdf = tract_boundaries.merge(wage_tax_df, on='tract_geoid', how='left', suffixes=('', '_wt'))
tract_gdf = tract_gdf.loc[:, ~tract_gdf.columns.str.endswith('_wt')]
tract_gdf = gpd.GeoDataFrame(tract_gdf, geometry='geometry', crs=tract_boundaries.crs or 'EPSG:4326')

print(f'{len(tract_gdf):,} tracts loaded')
print(tract_gdf[['agg_wage_income', 'agg_wage_income_moe', 'current_wage_tax']].describe())

In [ ]:
# LODES job-count cross-check (NOT a revenue estimator — see module docstring)
wac_df = fetch_lodes_wac(
    state_abbr='pa', year=LODES_YEAR,
    cache_path=str(DATA_DIR / f'pa_wac_{LODES_YEAR}.csv'),
)
lodes_summary = summarize_lodes_workplace_jobs(wac_df, county_fips=FIPS)
print('LODES workplace job-count cross-check (Philadelphia County):')
for k, v in lodes_summary.items():
    print(f'  {k}: {v:,.3f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
revenue_target = compute_wage_tax_revenue_target(
    wage_tax_df,
    resident_share=RESIDENT_REVENUE_SHARE,
    published_total_revenue=PUBLISHED_TOTAL_WAGE_TAX_REVENUE,
)
print(revenue_target)

# Plausibility check: back-of-envelope implied resident wage base vs. the ACS-modeled sum.
# $2.51B * 0.662 / 0.0375 ~= $44.3B implied aggregate resident wage income.
implied_resident_wage_base = PUBLISHED_TOTAL_WAGE_TAX_REVENUE * RESIDENT_REVENUE_SHARE / RESIDENT_WAGE_TAX_RATE
modeled_resident_wage_base = wage_tax_df['agg_wage_income'].sum()
base_gap_pct = (modeled_resident_wage_base / implied_resident_wage_base - 1) * 100
print(f'Implied resident wage base: ${implied_resident_wage_base:,.0f}   '
      f'Modeled (ACS sum): ${modeled_resident_wage_base:,.0f}   Gap: {base_gap_pct:+.1f}%')
assert abs(base_gap_pct) < 25.0, (
    f'Modeled resident wage base gap {base_gap_pct:.1f}% exceeds tolerance — check ACS_YEAR, '
    'B19062 variable, and RESIDENT_REVENUE_SHARE before proceeding.'
)

## Section 5 — Model the New Land-Only Tax

Fully-exempt parcels are held out of the solver (same norm as every other city notebook's Section 5) and not added back — they carry no signal and would only add a meaningless spike at the low end of the distribution.

In [ ]:
n_exempt = int((gdf['full_exmp'] == 1).sum())
gdf = gdf[gdf['full_exmp'] == 0].copy()

land_millage, new_revenue, gdf = model_land_only_tax(
    gdf,
    land_value_col='taxable_land',
    current_revenue=revenue_target['implied_total'],
)

print(f'Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, and charts).')
print(f'Land-only millage: {land_millage:.4f}  (current combined property tax millage: 13.998 for comparison)')
print(f'Revenue check: ${new_revenue:,.2f}  vs. target ${revenue_target["implied_total"]:,.2f}')
assert abs(new_revenue - revenue_target['implied_total']) / revenue_target['implied_total'] < 1e-6, (
    'Revenue-neutral solve did not converge exactly'
)

## Section 6 — Aggregate to Tract, Compute Winners and Losers

In [ ]:
gdf = match_to_census_tracts(gdf, tract_gdf[['tract_geoid', 'geometry']])
coverage_pct = gdf['tract_geoid'].notna().mean() * 100
print(f'Tract join coverage: {coverage_pct:.1f}%')

tract_land_tax = (
    gdf.groupby('tract_geoid', dropna=False)['new_tax'].sum()
    .reset_index().rename(columns={'new_tax': 'new_land_tax'})
)
tract_wide = tract_gdf.merge(tract_land_tax, on='tract_geoid', how='left')
tract_wide['new_land_tax'] = tract_wide['new_land_tax'].fillna(0.0)

# Wiring check: tract-level resident wage tax + the (non-geographic) implied non-resident
# total must exactly equal the revenue target passed into the land-only solve.
lhs = tract_wide['current_wage_tax'].sum() + revenue_target['implied_nonresident_total']
rhs = revenue_target['implied_total']
print(f'Wiring check: {lhs:,.2f} vs {rhs:,.2f}')
assert abs(lhs - rhs) < 1.0, 'resident tract sum + implied non-resident total should equal implied_total'

## Section 7 — Export + Report

Primary deliverable is the tract-level export (`save_wage_tax_swap_tract_export`); the parcel-level export is secondary detail on the new land tax only — `current_tax` is 0 for every parcel by construction (parcels don't pay the wage tax being eliminated).

In [ ]:
tract_export = save_wage_tax_swap_tract_export(
    tract_wide,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    land_millage=land_millage,
    model_type=MODEL_TYPE,
)

tract_report_gdf = tract_wide.merge(
    tract_export[['tract_geoid', 'net_change', 'net_change_pct']], on='tract_geoid', how='left',
)
tract_report_gdf = gpd.GeoDataFrame(tract_report_gdf, geometry='geometry', crs=tract_wide.crs)

parcel_export = save_wage_tax_swap_parcel_export(
    gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}_parcels.csv',
    land_millage=land_millage,
    land_value_col='taxable_land',
    parcel_id_col='parcel_number',
)

report = create_wage_tax_swap_report(
    tract_report_gdf,
    CITY_NAME,
    output_dir='../../analysis/reports',
    show=False,
    resident_wage_tax_total=revenue_target['modeled_resident_total'],
    nonresident_wage_tax_total=revenue_target['implied_nonresident_total'],
)
print(report)

n_tracts = len(tract_export)
n_winners = int((tract_export['net_change'] < 0).sum())  # net_change < 0: land tax < wage tax eliminated
n_losers = int((tract_export['net_change'] > 0).sum())
print()
print(f'{n_winners:,} of {n_tracts:,} tracts see a net decrease '
      f'(new land tax < eliminated wage tax); {n_losers:,} see a net increase.')
print(f"Resident wage tax eliminated: ${revenue_target['modeled_resident_total']:,.0f}")
print(f"Non-resident wage tax eliminated (mostly non-city residents): "
      f"${revenue_target['implied_nonresident_total']:,.0f}")
print(f'New land tax raised (borne entirely by city landowners): ${new_revenue:,.0f}')
print('Done.')

## Summary

This notebook models a genuinely different reform than the property-tax split-rate/abatement models elsewhere in this repo: it eliminates an entire tax *instrument* (the Wage & Earnings Tax) and replaces its revenue with a new, separate land-only levy, leaving the existing property tax untouched. Because there's no parcel-level wage data, the winners/losers comparison runs at census tract granularity, and mixes two different incidence populations (residents for the wage tax; landowners for the new land tax) — read the tract-level `net_change` as a directional, aggregate-fiscal-exposure signal, not a household-level guarantee.

The single largest structural finding is the commuter transfer (`incidence_shift.png`): roughly a third of the wage tax being eliminated is currently paid by non-resident commuters. A land-only tax cannot reach them — so eliminating the wage tax and replacing it entirely with a land tax shifts that share of Philadelphia's tax burden from (mostly suburban) commuters onto city landowners.